# S11 - Motores de reglas (Pagarium) — Soluciones

Cuaderno **de soluciones** de la S11 (RA5-b). Cada celda muestra la **salida real**. Se corresponde con `sesion11_motores_reglas.ipynb`.

## 1. La tabla DMN (recordatorio)

| Importe | Antigüedad (meses) | Intentos | Decisión |
|---|---|---|---|
| `< 500` | `>= 6` | `< 4` | Aprobar |
| `< 500` | `>= 6` | `>= 4` | Revisar |
| `>= 500` | — | — | Revisar |
| — | `< 6` | — | Rechazar |

Hit policy propuesta: **unique**.

## 2. `rule-engine`: primer ejemplo

In [1]:
import rule_engine

ctx = {"importe": 200, "n_intentos": 5, "antiguedad_meses": 10}
aprueba = rule_engine.Rule("importe < 500 and antiguedad_meses >= 6 and n_intentos < 4")
rechaza = rule_engine.Rule("n_intentos >= 4 or antiguedad_meses < 6")
print("aprueba:", aprueba.matches(ctx))
print("rechaza:", rechaza.matches(ctx))

aprueba: False
rechaza: True


## 3. Solución: la política completa (las 4 filas)

Se implementan las **4 reglas** de la tabla y se prueban **4 casos**, uno por fila. Después se muestra un **5º caso que produce solape**, para detectar el hueco de la tabla.

In [2]:
import rule_engine

reglas = {
    "aprobar":          rule_engine.Rule("importe < 500 and antiguedad_meses >= 6 and n_intentos < 4"),
    "revisar_intentos": rule_engine.Rule("importe < 500 and antiguedad_meses >= 6 and n_intentos >= 4"),
    "revisar_importe":  rule_engine.Rule("importe >= 500"),
    "rechazar":         rule_engine.Rule("antiguedad_meses < 6"),
}

def decidir(c):
    return [nombre for nombre, r in reglas.items() if r.matches(c)]

# 4 casos: uno por cada fila de la tabla
casos = [
    {"importe": 200, "antiguedad_meses": 10, "n_intentos": 2},  # fila 1 -> aprobar
    {"importe": 200, "antiguedad_meses": 10, "n_intentos": 5},  # fila 2 -> revisar (intentos)
    {"importe": 600, "antiguedad_meses": 10, "n_intentos": 2},  # fila 3 -> revisar (importe)
    {"importe": 200, "antiguedad_meses": 3,  "n_intentos": 2},  # fila 4 -> rechazar
]
for c in casos:
    print(c, "->", decidir(c))

# Caso de SOLAPE: importe >= 500 Y antiguedad < 6 -> engancha dos filas a la vez
solape = {"importe": 600, "antiguedad_meses": 3, "n_intentos": 2}
print("solape:", solape, "->", decidir(solape))

{'importe': 200, 'antiguedad_meses': 10, 'n_intentos': 2} -> ['aprobar']
{'importe': 200, 'antiguedad_meses': 10, 'n_intentos': 5} -> ['revisar_intentos']
{'importe': 600, 'antiguedad_meses': 10, 'n_intentos': 2} -> ['revisar_importe']
{'importe': 200, 'antiguedad_meses': 3, 'n_intentos': 2} -> ['rechazar']
solape: {'importe': 600, 'antiguedad_meses': 3, 'n_intentos': 2} -> ['revisar_importe', 'rechazar']


**Qué se aprende:** las cuatro filas devuelven una única decisión, pero el caso `importe >= 500` **y** `antiguedad < 6` engancha **dos reglas** (`revisar_importe` y `rechazar`). Eso viola la hit policy `unique`: es un **solape** que el verificador de cobertura debería detectar. La corrección es decidir el orden (p. ej. que `rechazar` tenga prioridad) o deshacer la ambigüedad en la tabla.

## 4. GoRules ZEN (grafo corregido)

La API de `zen-engine` cambió: con la versión actual (2.x) el grafo debe llevar `ref` en inputs/outputs y los valores de salida como **JSON string**. Este grafo **sí devuelve resultado**.

In [3]:
import json
from zen import ZenEngine

graph = {
  "contentType": "application/vnd.gorules.decision",
  "nodes": [
    {"id": "1", "name": "input", "type": "inputNode", "position": {"x": 0, "y": 0}},
    {"id": "2", "name": "tabla", "type": "decisionTableNode", "position": {"x": 300, "y": 0},
     "content": {
       "hitPolicy": "first",
       "inputs":  [{"id": "in.importe", "field": "importe", "name": "Importe", "ref": ""}],
       "outputs": [{"id": "out.decision", "field": "decision", "name": "Decision", "ref": ""}],
       "rules": [
         {"_id": "r1", "in.importe": "< 500",  "out.decision": "\"aprobar\""},
         {"_id": "r2", "in.importe": ">= 500", "out.decision": "\"revisar\""},
       ],
     }},
    {"id": "3", "name": "output", "type": "outputNode", "position": {"x": 600, "y": 0},
     "content": {"fields": [{"id": "out.decision", "field": "decision", "name": "Decision"}]}},
  ],
  "edges": [
    {"id": "e1", "sourceId": "1", "targetId": "2"},
    {"id": "e2", "sourceId": "2", "targetId": "3"},
  ],
}

engine = ZenEngine()
decision = engine.create_decision(json.dumps(graph))
print(decision.evaluate(json.dumps({"importe": 200})))
print(decision.evaluate(json.dumps({"importe": 600})))

{'performance': '287.4µs', 'result': {'decision': 'aprobar'}}
{'performance': '11.9µs', 'result': {'decision': 'revisar'}}


## 5. Benchmark: ¿compensa un motor?

Se mide el mismo caso con `if/else` y con `rule-engine` y se calcula el factor.

In [4]:
import timeit, rule_engine

ctx = {"importe": 200, "n_intentos": 5, "antiguedad_meses": 10}

def ifelse(c):
    if c["importe"] < 500 and c["antiguedad_meses"] >= 6 and c["n_intentos"] < 4:
        return "aprobar"
    if c["n_intentos"] >= 4 or c["antiguedad_meses"] < 6:
        return "rechazar"
    return "revisar"

regla = rule_engine.Rule("(importe < 500 and antiguedad_meses >= 6 and n_intentos < 4) or (n_intentos >= 4 or antiguedad_meses < 6)")

n = 100000
t_if = timeit.timeit(lambda: ifelse(ctx), number=n) / n * 1e6
t_rule = timeit.timeit(lambda: regla.matches(ctx), number=n) / n * 1e6
print(f"if/else:      {t_if:.3f} us")
print(f"rule-engine:  {t_rule:.3f} us")
print(f"factor:       {t_rule / t_if:.0f}x")

if/else:      0.075 us
rule-engine:  7.510 us
factor:       100x


**Conclusión (benchmark):** `rule-engine` es cientos de veces más lento que un `if/else`. Un motor compensa solo con **muchas reglas que cambian a menudo y las gestiona negocio** (BRMS); con 4 condiciones fijas, el `if/else` gana.